In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
 
CSV_PATH = "synthetic_cgpa_practice_data.csv"
TARGET = "Projected_CGPA"
SEED = 42

In [ ]:
df = pd.read_csv(CSV_PATH).drop_duplicates()

In [ ]:
ORDINALS = {
    "Level": ["100L", "200L", "300L", "400L", "500L"],
    "Study_Hours_Per_Week": ["Less than 5 hours", "5-9 hours", "10-14 hours",
                             "15-19 hours", "20-29 hours", "30 hours or more"],
    "Class_Attendance": ["Less than 50%", "50-69%", "70-89%", "90-100%"],
    "Continuous_Assessment_Score": ["Below 40%", "40-49%", "50-59%", "60-69%",
                                    "70-79%", "80% or above"],
    "Past_Questions_Practice": ["Never", "Rarely", "Occasionally", "Weekly", "Almost every day"],
    "Study_Groups_Participation": ["Never", "Rarely", "Sometimes", "Often", "Very often"],
    "Sleep_Hours_Per_Night": ["Less than 4 hours", "4-5 hours", "6-7 hours",
                              "8-9 hours", "10 hours or more"],
    "Extracurricular_Hours_Per_Week": ["0 hour", "1-5 hours", "6-10 hours",
                                       "11-15 hours", "More than 15 hours"],
    "Work_Hustle": ["No", "Yes, less than 5 hours per week", "Yes, 5-10 hours per week",
                    "Yes, 11-20 hours per week", "Yes, more than 20 hours per week"],
}
for col, order in ORDINALS.items():
    df[col] = df[col].map({v: i for i, v in enumerate(order)})
 
df["Accommodation"] = (df["Accommodation"].str.startswith("Off")).astype(int)  # 1 = off-campus
 
# Real Google Form data: CGPA arrives as text, so force numeric and drop impossible values
for col in ["Current_CGPA", "Last_SGPA", TARGET]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df.loc[(df[col] < 0) | (df[col] > 5), col] = np.nan
df = df.dropna(subset=[TARGET, "Current_CGPA"])

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nSummary stats:\n", df.describe().round(2).T)
 
num_cols = [c for c in df.columns if c not in ("Department",)]
corr = df[num_cols].corr()
print("\nCorrelation with target:\n", corr[TARGET].drop(TARGET).sort_values(ascending=False).round(2))
 
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df[TARGET].hist(bins=20, ax=axes[0]); axes[0].set_title("Projected CGPA distribution")
df.boxplot(column=["Current_CGPA", "Last_SGPA", TARGET], ax=axes[1]); axes[1].set_title("Outlier check")
plt.tight_layout(); plt.savefig("eda_distributions.png", dpi=150); plt.close()
 
plt.figure(figsize=(9, 7))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7)
plt.yticks(range(len(corr)), corr.columns, fontsize=7)
plt.title("Correlation heatmap"); plt.tight_layout()
plt.savefig("eda_correlation.png", dpi=150); plt.close()
 
# Outliers via IQR on the numeric columns
for c in ["Current_CGPA", "Last_SGPA", "Current_Credit_Load", "Carryovers"]:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    n_out = ((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum()
    print(f"Outliers in {c}: {n_out}")

Shape: (200, 16)

Missing values:
 Last_SGPA    8
dtype: int64

Summary stats:
 

                                count   mean   std    min    25%    50%  \
Level                           200.0   1.85  1.18   0.00   1.00   2.00   
Current_CGPA                    200.0   3.32  0.67   1.54   2.88   3.26   
Last_SGPA                       192.0   3.32  0.78   1.34   2.72   3.22   
Current_Credit_Load             200.0  22.22  3.72  14.00  19.00  22.00   
Carryovers                      200.0   0.60  0.81   0.00   0.00   0.00   
Study_Hours_Per_Week            200.0   2.26  1.26   0.00   1.00   2.00   
Class_Attendance                200.0   1.93  0.82   0.00   1.00   2.00   
Continuous_Assessment_Score     200.0   2.99  1.19   0.00   2.00   3.00   
Past_Questions_Practice         200.0   2.04  1.10   0.00   1.00   2.00   
Study_Groups_Participation      200.0   1.60  1.18   0.00   1.00   2.00   
Sleep_Hours_Per_Night           200.0   2.17  0.90   0.00   2.00   2.00   
Extracurricular_Hours_Per_Week  200.0   1.14  0.88   0.00   0.00   1.00   
Work_Hustle              

In [ ]:
X, y = df.drop(columns=[TARGET]), df[TARGET]
cat_cols = ["Department"]
num_feats = [c for c in X.columns if c not in cat_cols]
 
def make_pipeline(model, scale):
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    pre = ColumnTransformer([
        ("num", Pipeline(num_steps), num_feats),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    return Pipeline([("pre", pre), ("model", model)])
 
models = {
    "Linear Regression": make_pipeline(LinearRegression(), scale=True),
    "Random Forest": make_pipeline(
        RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=SEED), scale=False),
}

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
rows = []
for name, pipe in models.items():
    cvr = cross_validate(pipe, X, y, cv=cv,
                         scoring=("neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"))
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    rows.append({
        "Model": name,
        "CV MAE": -cvr["test_neg_mean_absolute_error"].mean(),
        "CV RMSE": -cvr["test_neg_root_mean_squared_error"].mean(),
        "CV R2": cvr["test_r2"].mean(),
        "Test MAE": mean_absolute_error(y_te, pred),
        "Test RMSE": mean_squared_error(y_te, pred) ** 0.5,
        "Test R2": r2_score(y_te, pred),
    })
results = pd.DataFrame(rows).set_index("Model").round(3)
print("\nModel comparison:\n", results)
results.to_csv("model_comparison.csv")
 


Model comparison:
                    CV MAE  CV RMSE  CV R2  Test MAE  Test RMSE  Test R2
Model                                                                  
Linear Regression   0.153    0.188  0.934     0.140      0.185    0.945
Random Forest       0.166    0.210  0.913     0.168      0.205    0.932


In [ ]:
lr = models["Linear Regression"]
feat_names = lr.named_steps["pre"].get_feature_names_out()
coefs = pd.Series(lr.named_steps["model"].coef_, index=feat_names).sort_values(key=abs, ascending=False)
print("\nLinear Regression coefficients (standardised):\n", coefs.head(10).round(3))
 
rf = models["Random Forest"]
perm = permutation_importance(rf, X_te, y_te, n_repeats=20, random_state=SEED, scoring="r2")
perm_imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print("\nRandom Forest permutation importance:\n", perm_imp.round(3))
perm_imp.sort_values().plot.barh(figsize=(7, 6), title="Permutation importance (RF)")
plt.tight_layout(); plt.savefig("feature_importance.png", dpi=150); plt.close()
 


Linear Regression coefficients (standardised):
 num__Current_CGPA                         0.425
num__Last_SGPA                            0.222
cat__Department_Computer Engineering     -0.132
cat__Department_Estate Management         0.113
cat__Department_Cyber Security           -0.092
cat__Department_Mechanical Engineering   -0.086
num__Class_Attendance                     0.077
cat__Department_Biochemistry              0.076
cat__Department_Architecture             -0.070
cat__Department_Electrical Engineering    0.069
dtype: float64

Random Forest permutation importance:
 Current_CGPA                      1.256
Last_SGPA                         0.066
Class_Attendance                  0.004
Carryovers                        0.003
Study_Groups_Participation        0.002
Level                             0.002
Extracurricular_Hours_Per_Week    0.001
Study_Hours_Per_Week              0.001
Accommodation                     0.001
Continuous_Assessment_Score       0.000
Past_Questions_P

In [ ]:
best_name = results["CV RMSE"].idxmin()          # pick on CV error, not on test set
final_model = models[best_name].fit(X, y)         # refit on all data
print(f"\nFinal model: {best_name}")
 
# Only features a student can realistically change. direction: +1 raise, -1 lower
ACTIONABLE = {
    "Study_Hours_Per_Week":          (+1, "Increase your weekly study hours"),
    "Class_Attendance":              (+1, "Attend more of your classes"),
    "Continuous_Assessment_Score":   (+1, "Aim for higher scores in assignments, quizzes and tests"),
    "Past_Questions_Practice":       (+1, "Practise past questions more often"),
    "Study_Groups_Participation":    (+1, "Join study groups / tutorials more regularly"),
    "Work_Hustle":                   (-1, "Reduce the time you spend on work/business"),
    "Extracurricular_Hours_Per_Week": (-1, "Cut back time on extracurricular activities"),
}
MAXLEVEL = {c: len(ORDINALS[c]) - 1 for c in ACTIONABLE}
 
def recommend(student: pd.Series, top_k=3):
    """student: one row with the same columns as X (already encoded)."""
    base = float(final_model.predict(student.to_frame().T)[0])
    gains = []
    for feat, (direction, advice) in ACTIONABLE.items():
        new_val = student[feat] + direction
        if new_val < 0 or new_val > MAXLEVEL[feat]:
            continue                                   # already at the limit
        trial = student.copy(); trial[feat] = new_val
        delta = float(final_model.predict(trial.to_frame().T)[0]) - base
        if delta > 0.005:
            gains.append((delta, advice, feat))
    gains.sort(reverse=True)
    print(f"\nPredicted CGPA: {min(base, 5.0):.2f}")
    if not gains:
        print("You're already on a strong track. Keep your current habits.")
    for delta, advice, feat in gains[:top_k]:
        print(f"  -> {advice} (one step up/down on '{feat}' could add about {delta:+.2f} to your CGPA)")
    return base, gains[:top_k]
 
# Demo: run it on a few existing students
for idx in X.sample(3, random_state=SEED).index:
    recommend(X.loc[idx])
 


Final model: Linear Regression

Predicted CGPA: 3.34
  -> Attend more of your classes (one step up/down on 'Class_Attendance' could add about +0.11 to your CGPA)
  -> Increase your weekly study hours (one step up/down on 'Study_Hours_Per_Week' could add about +0.03 to your CGPA)
  -> Reduce the time you spend on work/business (one step up/down on 'Work_Hustle' could add about +0.03 to your CGPA)

Predicted CGPA: 2.54
  -> Attend more of your classes (one step up/down on 'Class_Attendance' could add about +0.11 to your CGPA)
  -> Cut back time on extracurricular activities (one step up/down on 'Extracurricular_Hours_Per_Week' could add about +0.04 to your CGPA)
  -> Increase your weekly study hours (one step up/down on 'Study_Hours_Per_Week' could add about +0.03 to your CGPA)

Predicted CGPA: 4.90
  -> Cut back time on extracurricular activities (one step up/down on 'Extracurricular_Hours_Per_Week' could add about +0.04 to your CGPA)
  -> Practise past questions more often (one step u